# LogisChain AI — 05: Model Evaluation & Explainability

Comprehensive evaluation of all models with SHAP explainability and business-metric reporting.

**Goals:**
- Full classification report with KS, Gini, IV
- SHAP global and local explanations
- Supply chain vs financial contribution decomposition
- LogisChain Lab simulation run and scoring

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.models.xgboost_model import XGBoostRiskModel
from src.utils.metrics import classification_report_dict, ks_statistic, gini_coefficient, information_value
from src.utils.explainability import LogisChainExplainer
from src.simulation.game_modes import GameSession
from src.simulation.scoring import compute_final_grade

gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
fp = FeaturePipeline()
fused = fp.run(data['carriers'], data['shipments'], data['financial'])

In [ ]:
# Train and evaluate XGBoost
from sklearn.model_selection import train_test_split
target = 'carrier_failure' if 'carrier_failure' in fused.columns else 'default_flag'
drop_cols = [target, 'carrier_id', 'company_id', 'carrier_type', 'region', 'industry']
X = fused.drop(columns=[c for c in drop_cols if c in fused.columns]).select_dtypes(include=np.number).fillna(0)
y = fused[target].fillna(0) if target in fused.columns else pd.Series(np.zeros(len(fused)))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = XGBoostRiskModel(config={'n_estimators': 200})
model.fit(X_train, y_train, X_test, y_test)
probs = model.predict_proba(X_test)

report = classification_report_dict(y_test.values, probs)
print('Full Evaluation Report:')
for k, v in report.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

In [ ]:
# SHAP explainability
explainer = LogisChainExplainer(model.model, model_type='tree')
explainer.fit(X_train)
global_imp = explainer.global_importance(X_test)
print('Top 10 Global Feature Importances (SHAP):')
print(global_imp.head(10).to_string(index=False))
decomp = explainer.sc_vs_financial_decomposition(X_test)
print(f'\nSC contribution: {decomp["supply_chain_shap_pct"]:.1f}%')
print(f'Financial contribution: {decomp["financial_shap_pct"]:.1f}%')

In [ ]:
# LogisChain Lab simulation
print('=== LogisChain Lab: Campaign Mode Simulation ===')
session = GameSession('campaign_asia_pacific', seed=42)
for period in range(8):
    actions = []
    if period % 2 == 0:
        actions.append(('buy_insurance', {'coverage_pct': 0.15}))
    if period % 3 == 0:
        actions.append(('build_credit_reserves', {'amount_usd': 100_000}))
    result = session.play_period(actions or [('hold', {})])
    print(f'Period {period+1}: Score={result.period_score:.0f} | Scenario={result.scenario_applied or "None"} | Loss=${result.financial_impact_usd:,.0f}')

final = session.get_leaderboard_entry()
grade = compute_final_grade(final['final_score'], mode_target=2000)
print(f'\nFinal Grade: {grade["grade"]} ({grade["rank"]})')
print(f'Score: {grade["total_score"]} / {grade["target_score"]} ({grade["achievement_pct"]}%)')
print(f'Feedback: {grade["feedback"]}')